In [1]:
import pandas as pd

Importation des données de toutes les élections pour élections municipales

In [2]:
all_elections = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/candidats_results.txt", sep=";")

/tmp/ipykernel_9822/2459301605.py:5: DtypeWarning: Columns (2,4,6,7,8,12,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  all_elections = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/candidats_results.txt", sep=";")


In [3]:
donnees_restreintes_muni = all_elections[all_elections["id_election"].str.contains("muni", case=False, na=False)]
donnees_restreintes_muni.head()

,id_election,id_brut_miom,Code du département,Code de la commune,Code du b.vote,N°Panneau,Libellé Abrégé Liste,Libellé Etendu Liste,Nom Tête de Liste,Voix,% Voix/Ins,% Voix/Exp,Sexe,Nom,Prénom,Nuance,Binôme,Liste
4267946,2020_muni_t2,01012_0001,01,12,0001,7.0,NaN,NaN,NaN,0.0,0.00,0.00,M,PIRES,Hervé,NC,NaN,NaN
4267947,2020_muni_t2,01034_0001,01,34,0001,1.0,NaN,NaN,NaN,202.0,22.65,44.30,M,FOGNINI,Jean-Marc,LDIV,NaN,REUNIR POUR BELLEY AVEC JEAN MARC FOGNINI
4267948,2020_muni_t2,01034_0002,01,34,0002,1.0,NaN,NaN,NaN,235.0,24.15,48.76,M,FOGNINI,Jean-Marc,LDIV,NaN,REUNIR POUR BELLEY AVEC JEAN MARC FOGNINI
4267949,2020_muni_t2,01034_0003,01,34,0003,1.0,NaN,NaN,NaN,257.0,26.58,62.99,M,FOGNINI,Jean-Marc,LDIV,NaN,REUNIR POUR BELLEY AVEC JEAN MARC FOGNINI
4267950,2020_muni_t2,01034_0004,01,34,0004,1.0,NaN,NaN,NaN,168.0,17.34,36.68,M,FOGNINI,Jean-Marc,LDIV,NaN,REUNIR POUR BELLEY AVEC JEAN MARC FOGNINI


In [4]:
# ---- Étape 1 : calcul des voix totales par élection et ville ----
voix_totales = (
    donnees_restreintes_muni
    .assign(ident_election_ville=lambda d: d["id_brut_miom"].str[:5])
    .groupby(["id_election", "ident_election_ville"], as_index=False)
    .agg(voix_total_ville_elec=("Voix", "sum"))
)

In [5]:

# ---- Étape 2 : calcul des voix par candidat ----
donnees_muni_long = (
    donnees_restreintes_muni
    .assign(ident_election_ville=lambda d: d["id_brut_miom"].str[:5])
    .groupby(["id_election", "ident_election_ville", "Nom", "Prénom", "Nuance"], as_index=False)
    .agg(voix_totales_candidat=("Voix", "sum"))
    .merge(voix_totales, on=["id_election", "ident_election_ville"], how="left")
    .assign(voix_pct=lambda d: d["voix_totales_candidat"] / d["voix_total_ville_elec"] * 100)
    .sort_values(["id_election", "ident_election_ville", "voix_pct"], ascending=[True, True, False])
)

# ---- Étape 3 : classement des deux premiers ----
donnees_muni_wide = (
    donnees_muni_long
    .assign(rang=lambda d: d.groupby(["id_election", "ident_election_ville"]).cumcount() + 1)
    .query("rang <= 2")
)

# ---- Étape 4 : passage en format large ----
donnees_muni_wide = (
    donnees_muni_wide
    .pivot(index=["id_election", "ident_election_ville"],
           columns="rang",
           values=["Prénom", "Nom", "Nuance","voix_pct"])
)

# Aplatir les noms de colonnes hiérarchiques comme "rang1_Prénom"
donnees_muni_wide.columns = [
    f"rang{r}_{v}" for v, r in donnees_muni_wide.columns
]
donnees_muni_wide = donnees_muni_wide.reset_index()

# ---- Étape 5 : extraire les variables election / tour ----
donnees_muni_wide = (
    donnees_muni_wide
    .assign(
        election=lambda d: d["id_election"].str[:9],
        tour=lambda d: d["id_election"].str[10:12])
    .groupby(["election", "ident_election_ville"], group_keys=False)
    .filter(lambda g: not ("t1" in g["tour"].values and "t2" in g["tour"].values and g["tour"].eq("t1").any()))
)


In [6]:
donnees_muni_long.head()

,id_election,ident_election_ville,Nom,Prénom,Nuance,voix_totales_candidat,voix_total_ville_elec,voix_pct
1,2008_muni_t1,01004,EXPOSITO,Josiane,LUG,1341.0,4272.0,31.390449
0,2008_muni_t1,01004,CASTELLANO,Sandrine,LDVD,1327.0,4272.0,31.062734
2,2008_muni_t1,01004,PAVIER,Bernard,LGC,820.0,4272.0,19.194757
3,2008_muni_t1,01004,SASSO,Jean-Marc,LDVD,784.0,4272.0,18.352060
5,2008_muni_t1,01014,MAISSIAT,Liliane,LMAJ,985.0,1251.0,78.737010


In [7]:
#on prépare la suite en créant un fichier listant les têtes de liste des municipales et leur couleur politique
liste_candidats_nuance = (
                    donnees_muni_long
                    .assign(annee = lambda d : d["id_election"].str[1:4])
                    .loc[:, ["annee", "Nom", "Prénom", "Nuance"]]
)
liste_candidats_nuance.head()

,annee,Nom,Prénom,Nuance
1,008,EXPOSITO,Josiane,LUG
0,008,CASTELLANO,Sandrine,LDVD
2,008,PAVIER,Bernard,LGC
3,008,SASSO,Jean-Marc,LDVD
5,008,MAISSIAT,Liliane,LMAJ


Importation des données des intercommunalités

In [ ]:
grp_2025 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/liste-des-groupements-france-entiere-20250127.csv", encoding="latin-1", sep = ";")
grp_2019 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/Liste_des_groupements_en_2019.csv", encoding="latin-1", sep = ";")
grp_2013 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/Liste_des_groupements_en_2013.csv", encoding="latin-1", sep = ";")
